In [32]:
import httpx
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
import time
import json
import psycopg
import os
import sys
import numpy as np
from dotenv import load_dotenv
from itertools import zip_longest

sys.path.append(os.path.abspath('./src'))
import db_functions as dbf
load_dotenv()
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST", "localhost")
port = os.getenv("DB_PORT", "5432")
dbname = os.getenv("DB_NAME")
conn_str = f"postgresql://{user}:{password}@{host}:{port}/{dbname}"
api_key = os.getenv("API_KEY")
api_url = 'https://api.stratz.com/graphql'
headers = {
    'User-Agent': 'STRATZ_API',
    "Authorization": f"Bearer {api_key}"
}

In [2]:
def query_stratz(query: str, variables={}):
    with httpx.Client(headers=headers) as client:
        response = client.post(
            url=api_url,
            json={'query': query, 'variables': variables}
        )
        result = response.json()
        if "errors" in result:
            raise Exception(f"GraphQL Error: {result['errors']}")
        return result

In [ ]:
# Get game versions
query = """
    query {
        constants {
            gameVersions {
                id
                name
                asOfDateTime
            }
        }
    }
"""
result = query_stratz(query)
df = pd.DataFrame(result['data']['constants']['gameVersions'])
df['asOfDateTime'] = df['asOfDateTime'].apply(datetime.fromtimestamp)
dbf.create_table_from_df(df, 'patches', conn_str)
dbf.insert_df_into_table(df, 'patches', conn_str)

In [3]:
query = """
    query($gameVersionId: Short!) {
        constants {
            npcs(gameVersionId: $gameVersionId) {
                id
                name
                stat {
                    statusHealth
                    statusHealthRegen
                    attackDamageMin
                    attackDamageMax
                    attackRate
                    attackRange
                    movementSpeed
                    isNeutralUnitType
                    isAncient
                    teamName
                }
            }
        }
    }
"""
variables = {'gameVersionId': 182} #TODO: replace hardcoded value
result = query_stratz(query, variables)

In [4]:
df = pd.DataFrame(result['data']['constants']['npcs'])
stats_df = pd.json_normalize(df['stat'])
df = df.join(stats_df).drop(columns=['stat'])
discard_patterns = [
    'thinker', 'companion', 'visual', 'sound', 'event', 
    'shmup', 'banana', 'target_dummy', 'looping', 'promo'
]
discard_regex = '|'.join(discard_patterns)
df_filtered = df[~df['name'].str.contains(discard_regex, case=False, na=False)]
df_filtered.dtypes
df_filtered.convert_dtypes(convert_integer=False).dtypes
df_filtered
dbf.create_table_from_df(df_filtered, 'npcs', conn_str, False)
dbf.insert_df_into_table(df_filtered, 'npcs', conn_str)

Table 'npcs' created successfully.
Data inserted into table 'npcs' successfully.


In [ ]:
query = """
    query($id: Long!) {
      match(id: $id) {
        id
        tournamentId
        tournamentRound
        leagueId
        radiantTeamId
        direTeamId
        seriesId
        gameVersionId
        regionId
        clusterId
        didRadiantWin
        startDateTime
        endDateTime
        durationSeconds
        firstBloodTime
        towerStatusRadiant
        towerStatusDire
        barracksStatusRadiant
        barracksStatusDire
        rank
        actualRank
        averageRank
        averageImp
        bracket
        analysisOutcome
        topLaneOutcome
        midLaneOutcome
        bottomLaneOutcome
        predictedOutcomeWeight
        pickBans {
          isPick
          heroId
          order
          isRadiant
        }
        chatEvents {
          time
          type
          fromHeroId
          toHeroId
          value
          pausedTick
          isRadiant
        }
        predictedWinRates
        winRates
        radiantNetworthLeads
        radiantExperienceLeads
        radiantKills
        direKills
        towerDeaths {
          time
          npcId
          isRadiant
          attacker
        }
        towerStatus {
          towers {
            npcId
            hp
          }
          outposts {
            npcId
            isControlledByRadiant
            isRadiantSide
          }
        }
        players {
          heroId
          isRadiant
          isVictory
          variant
          imp
          lane
          position
          networth
          goldPerMinute
          goldSpent
          towerDamage
          heroDamage
          intentionalFeeding
          stats {
            impPerMinute
            goldPerMinute
            networthPerMinute
            experiencePerMinute
            towerDamagePerMinute
            campStack
            deathEvents {
              time
              attacker
              isDieBack
            }
            farmDistributionReport {
              creepLocation {
                id
                gold
              }
              neutralLocation {
                id
                gold
              }
              ancientLocation {
                id
                gold
              }
              buildings {
                id
                gold
              }
              bountyGold {
                id
                gold
              }
              other {
                id
                gold
              }
              buyBackGold
            }
            matchPlayerBuffEvent {
              time
              abilityId
              itemId
              stackCount
            }
            inventoryReport {
              item0 {
                itemId
              }
              item1 {
                itemId
              }
              item2 {
                itemId
              }
              item3 {
                itemId
              }
              item4 {
                itemId
              }
              item5 {
                itemId
              }
              neutral0 {
                itemId
              }
            }
            itemPurchases {
              time
              itemId
            }
            courierKills {
              time
            }
            runes {
              time
              rune
              action
              positionX
              positionY
            }
            wards {
              time
              type
              positionX
              positionY
            }
            wardDestruction {
              time
              gold
              isWard
            }
          }
        }
      }
    }
    """
#TODO: isVictory possibly redundant in each player
variables = {'id': 8110922237}
result = query_stratz(query, variables)
result_json = result['data']['match']
#TODO: find first blood team by looking for chat event type 5
#TODO: both hero IDs present: tip
#TODO: find out what is roshan kills etc. from an actual replay

In [15]:
result_json

{'id': 8110922237,
 'tournamentId': None,
 'tournamentRound': None,
 'leagueId': 17628,
 'radiantTeamId': 350190,
 'direTeamId': 9646073,
 'seriesId': 938762,
 'gameVersionId': 178,
 'regionId': 13,
 'clusterId': 236,
 'didRadiantWin': True,
 'startDateTime': 1735889233,
 'endDateTime': 1735891040,
 'durationSeconds': 1807,
 'firstBloodTime': 18,
 'towerStatusRadiant': 1974,
 'towerStatusDire': 1028,
 'barracksStatusRadiant': 63,
 'barracksStatusDire': 3,
 'rank': 80,
 'actualRank': 80,
 'averageRank': None,
 'averageImp': -3,
 'bracket': 8,
 'analysisOutcome': 'COMEBACK',
 'topLaneOutcome': 'TIE',
 'midLaneOutcome': 'TIE',
 'bottomLaneOutcome': 'DIRE_VICTORY',
 'pickBans': [{'isPick': False, 'heroId': 114, 'order': 0, 'isRadiant': False},
  {'isPick': False, 'heroId': 69, 'order': 1, 'isRadiant': True},
  {'isPick': False, 'heroId': 99, 'order': 2, 'isRadiant': True},
  {'isPick': False, 'heroId': 110, 'order': 3, 'isRadiant': False},
  {'isPick': False, 'heroId': 38, 'order': 4, 'isR

In [ ]:
nested_vars = []
for key, value in result_json.items():
    if type(value) in [dict, list]:
        print(key, type(value))
        nested_vars.append(key)

pickBans <class 'list'>
chatEvents <class 'list'>
predictedWinRates <class 'list'>
winRates <class 'list'>
radiantNetworthLeads <class 'list'>
radiantExperienceLeads <class 'list'>
radiantKills <class 'list'>
direKills <class 'list'>
towerDeaths <class 'list'>
towerStatus <class 'list'>
players <class 'list'>


In [ ]:
cols_to_include = [
    'id',
    'tournamentId',
    'tournamentRound',
    'leagueId',
    'radiantTeamId',
    'direTeamId',
    'seriesId',
    'gameVersionId',
    'regionId',
    'clusterId',
    'didRadiantWin',
    'startDateTime',
    'endDateTime',
    'durationSeconds',
    'firstBloodTime',
    'towerStatusRadiant',
    'towerStatusDire',
    'barracksStatusRadiant',
    'barracksStatusDire',
    'rank',
    'actualRank',
    'averageRank',
    'averageImp',
    'bracket',
    'analysisOutcome',
    'topLaneOutcome',
    'midLaneOutcome',
    'bottomLaneOutcome',
    'predictedOutcomeWeight'
]

In [ ]:
filtered_match_details = {k: result_json[k] for k in cols_to_include}
df_match_details = pd.DataFrame([filtered_match_details])
df_pickbans = pd.DataFrame(result_json['pickBans'])
df_pickbans.insert(0, 'match_id', result_json['id'])
df_chatevents = pd.json_normalize(result_json['chatEvents'])
df_chatevents.insert(0, 'match_id', result_json['id'])
df_predicted_win_rates = pd.DataFrame({
    'match_id': result_json['id'],
    'predicted_win_rate': result_json['predictedWinRates']
})
df_win_rates = pd.DataFrame({
    'match_id': result_json['id'],
    'win_rates': result_json['winRates'], 
})
df_kills = pd.DataFrame({
    'match_id': result_json['id'],
    'radiant_kills': result_json['radiantKills'],
    'dire_kills': result_json['direKills']
})
df_leads = pd.DataFrame({
    'match_id': result_json['id'],
    'radiant_networth_leads': result_json['radiantNetworthLeads'],
    'radiant_experience_leads': result_json['radiantExperienceLeads']
})
df_tower_deaths = pd.json_normalize(result_json['towerDeaths'])
df_tower_deaths.insert(0, 'match_id', result_json['id'])
snapshots = []
tower_updates = []
outpost_updates = []
for i, snapshot in enumerate(result_json['towerStatus']):
    snapshot_id = f"{result_json['id']}_{i}"
    snapshots.append({
        'snapshot_id': snapshot_id,
        'match_id': result_json['id'],
        'order_index': i 
    })
    for t in snapshot['towers']:
        tower_updates.append({
            'snapshot_id': snapshot_id,
            'npc_id': t['npcId'],
            'hp': t['hp']
        })
        
    for o in snapshot['outposts']:
        outpost_updates.append({
            'snapshot_id': snapshot_id,
            'npc_id': o['npcId'],
            'is_radiant_controlled': o['isControlledByRadiant'],
            'is_radiant_side': o['isRadiantSide']
        })
df_snapshots = pd.DataFrame(snapshots)
df_tower_updates = pd.DataFrame(tower_updates)
df_outpost_updates = pd.DataFrame(outpost_updates)
df_players = pd.json_normalize(result_json['players'])
cols_to_keep = [
    'heroId',
    'isRadiant',
    'isVictory',
    'variant',
    'imp',
    'lane',
    'position',
    'networth',
    'goldPerMinute',
    'goldSpent',
    'towerDamage',
    'heroDamage',
    'intentionalFeeding'
]
df_players = df_players[cols_to_keep]

In [ ]:
df_pickbans
df_chatevents
df_predicted_win_rates
df_win_rates
df_kills
df_leads
df_tower_deaths
df_snapshots
df_tower_updates
df_outpost_updates
df_players

In [ ]:
for key, value in result_json['players'][0].items():
    if type(value) in [dict, list]:
        print(key, type(value))
        nested_vars.append(key)

stats <class 'dict'>


In [25]:
for key, value in result_json['players'][0]['stats'].items():
    if type(value) in [dict, list]:
        print(key, type(value))
        nested_vars.append(key)

impPerMinute <class 'list'>
goldPerMinute <class 'list'>
networthPerMinute <class 'list'>
experiencePerMinute <class 'list'>
towerDamagePerMinute <class 'list'>
campStack <class 'list'>
deathEvents <class 'list'>
farmDistributionReport <class 'dict'>
matchPlayerBuffEvent <class 'list'>
inventoryReport <class 'list'>
itemPurchases <class 'list'>
courierKills <class 'list'>
runes <class 'list'>
wards <class 'list'>
wardDestruction <class 'list'>


In [195]:
performance_metrics_columns = [
    'match_id',
    'hero_id',
    'minute',
    'imp_per_minute',
    'gold_per_minute',
    'networth_per_minute',
    'experience_per_minute',
    'tower_damage_per_minute',
    'camp_stack'
]
df_performance_metrics = pd.DataFrame(columns=performance_metrics_columns)
df_death_events = pd.DataFrame(columns=['match_id', 'hero_id', 'time', 'attacker', 'isDieBack'])
df_farm = pd.DataFrame(columns=['match_id', 'hero_id', 'source_type', 'id',	'gold'])
df_courier_kills = pd.DataFrame(columns=['match_id', 'hero_id', 'time'])
df_runes = pd.DataFrame(columns=['match_id', 'hero_id', 'time', 'rune', 'action', 'positionX', 'positionY'])
df_wards = pd.DataFrame(columns=[
    'match_id',
    'hero_id',
    'time',
    'type',
    'positionX',
    'positionY'
])
df_ward_destructions = pd.DataFrame(columns=[
    'match_id',
    'hero_id',
    'time',
    'gold',
    'isWard'
])
df_inventory_reports = pd.DataFrame(columns=[
    'match_id',
    'hero_id',
    'minute',
    'item0_id',
    'item1_id',
    'item2_id',
    'item3_id',
    'item4_id',
    'item5_id',
    'neutral0_id',
])
#TODO: matchplayerbuffevent, inventoryreport
df_purchases = pd.DataFrame(columns=['match_id', 'hero_id', 'time', 'itemId'])
df_buffs = pd.DataFrame(columns=[
    'match_id',
    'hero_id',
    'time',
    'abilityId',
    'itemId',
    'stackCount'
])
for idx, player in enumerate(result_json['players']):
    stats = player['stats'] 

    df_ir = pd.json_normalize(stats['inventoryReport'])
    existing_item_cols = [c for c in df_ir.columns if '.itemId' in c]
    df_ir = df_ir[existing_item_cols].copy()
    df_ir = df_ir.rename(columns=lambda x: x.replace('.itemId', '_id'))
    df_ir.insert(0, 'hero_id', player['heroId'])
    df_ir.insert(0, 'match_id', result_json['id'])
    df_inventory_reports = pd.concat([df_inventory_reports, df_ir])

    df_pm = pd.DataFrame({
        'match_id': result_json['id'],
        'hero_id': player['heroId'],
        'minute': range(len(stats['impPerMinute'])),
        'imp_per_minute': pd.Series(stats['impPerMinute']),
        'gold_per_minute': pd.Series(stats['goldPerMinute']),
        'networth_per_minute': pd.Series(stats['networthPerMinute']),
        'experience_per_minute': pd.Series(stats['experiencePerMinute']),
        'tower_damage_per_minute': pd.Series(stats['towerDamagePerMinute']),
        'camp_stack': pd.Series(stats['campStack'])
    })
    df_performance_metrics = pd.concat([df_performance_metrics, df_pm])
    df_matchid_heroid = pd.DataFrame({
        'match_id': result_json['id'],
        'hero_id': player['heroId']
    }, index=range(len(stats['deathEvents'])))
    df_de = pd.DataFrame(stats['deathEvents'])
    df_death_events = pd.concat([df_death_events, pd.concat([df_matchid_heroid, df_de], axis=1)])
    rows = []
    for category, content in stats['farmDistributionReport'].items():
        if isinstance(content, list):
            for entry in content:
                # Copy entry so we don't modify the original JSON
                row = entry.copy()
                row['source_type'] = category
                rows.append(row)
        elif isinstance(content, dict):
            row = content.copy()
            row['source_type'] = category
            rows.append(row)
    df_f = pd.DataFrame(rows)
    df_f.insert(0, 'hero_id', player['heroId'])
    df_f.insert(0, 'match_id', result_json['id'])
    df_farm = pd.concat([df_farm, df_f])
    buff_rows = []
    for buff in stats['matchPlayerBuffEvent']:
        buff_rows.append({
            'time': buff['time'],
            'item_id': buff['itemId'],
            'ability_id': buff['abilityId']
        })
    df_b = pd.DataFrame(buff_rows)
    df_b.insert(0, 'hero_id', player['heroId'])
    df_b.insert(0, 'match_id', result_json['id'])
    df_buffs = pd.concat([df_buffs, df_b])

    df_p = pd.DataFrame(stats['itemPurchases'])
    df_p.insert(0, 'hero_id', player['heroId'])
    df_p.insert(0, 'match_id', result_json['id'])
    df_purchases = pd.concat([df_purchases, df_p])
    df_c = pd.DataFrame(stats['courierKills'])
    df_c.insert(0, 'hero_id', player['heroId'])
    df_c.insert(0, 'match_id', result_json['id'])
    df_courier_kills = pd.concat([df_courier_kills, df_c])
    df_r = pd.DataFrame(stats['runes'])
    df_r.insert(0, 'hero_id', player['heroId'])
    df_r.insert(0, 'match_id', result_json['id'])
    df_runes = pd.concat([df_runes, df_r])
    df_w = pd.DataFrame(stats['wards'])
    df_w.insert(0, 'hero_id', player['heroId'])
    df_w.insert(0, 'match_id', result_json['id'])
    df_wards = pd.concat([df_wards, df_w])
    df_wd = pd.DataFrame(stats['wardDestruction'])
    df_wd.insert(0, 'hero_id', player['heroId'])
    df_wd.insert(0, 'match_id', result_json['id'])
    df_ward_destructions = pd.concat([df_ward_destructions, df_wd])
df_death_events = df_death_events.reset_index().drop(['index'], axis=1)

C:\Users\benib\AppData\Local\Temp\ipykernel_13572\2702656196.py:63: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_inventory_reports = pd.concat([df_inventory_reports, df_ir])
C:\Users\benib\AppData\Local\Temp\ipykernel_13572\2702656196.py:76: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_performance_metrics = pd.concat([df_performance_metrics, df_pm])
C:\Users\benib\AppData\Local\Temp\ipykernel_13572\2702656196.py:130: FutureWarning: The behavior of DataFrame concatenation with empty or al

In [ ]:
#TODO: add buybackgold to players_df